# Unit 1: Root Finding II - The Need for Speed
**Date:** Jan 22 (Thursday)
**Topic:** Newton-Raphson Method & Scipy Optimization

## 1. Introduction: Slopes vs. Brackets
In the last lecture, we used the **Bisection Method**. It is reliable (guaranteed to converge) but slow. It blindly chops the interval in half, ignoring the *shape* of the function.

Today, we learn the **Newton-Raphson Method** (often just "Newton's Method").
* **Idea:** If we know the slope (derivative) of the function, we can predict where it will hit zero.
* **Analogy:** Bisection is like feeling your way in the dark. Newton's method is like using a flashlight to see where the path (the tangent line) leads.



### The Geometry
Instead of a midpoint, we draw a **tangent line** at our current guess $x_0$. We find where this tangent line crosses the x-axis, and that becomes our new guess $x_1$.


## 2. The Formula (Derivation)
The equation for a tangent line at point $(x_n, f(x_n))$ with slope $f'(x_n)$ is:
$$y - f(x_n) = f'(x_n) (x - x_n)$$

We want to find the $x$ where $y = 0$ (the x-intercept). Let's call this new point $x_{n+1}$.
$$0 - f(x_n) = f'(x_n) (x_{n+1} - x_n)$$

Rearranging for $x_{n+1}$, we get the famous **Newton-Raphson Update Formula**:

$$x_{n+1} = x_n - \frac{f(x_n)}{f'(x_n)}$$

### Key Differences from Bisection
| Feature | Bisection | Newton-Raphson |
| :--- | :--- | :--- |
| **Input** | Interval $[a, b]$ | Single guess $x_0$ |
| **Requirements** | Continuous function | Differentiable function ($f'$) |
| **Speed** | Linear (Slow) | Quadratic (Very Fast) |
| **Stability** | Guaranteed to work | Can fail (diverge) |

## Problem 1: Manual Implementation
**Goal:** Implement the Newton-Raphson step manually.

**Task:** Find the root of $f(x) = x^2 - 612$. (i.e., calculate $\sqrt{612}$).
* $f(x) = x^2 - 612$
* $f'(x) = 2x$

In [ ]:
import numpy as np

def f(x):
    return x**2 - 612

def df(x):
    return 2*x

# Starting Guess (Pick something reasonable)
x = 10.0
print(f"Start: {x}")

# TODO: Implement the update formula inside the loop
# Formula: x_new = x - f(x) / f'(x)
for i in range(8):
    x = x - (f(x)/df(x))
    print(f"Iteration {i+1}: {x}")

# --- CHECK ---
print(f"\nActual Sqrt(612): {np.sqrt(612)}")
# Notice how fast it converges! By iteration 3 or 4, it's usually perfect.

Start: 10.0
Iteration 1: 35.6
Iteration 2: 26.395505617977527
Iteration 3: 24.790635492455475
Iteration 4: 24.738688294075324
Iteration 5: 24.738633753766084
Iteration 6: 24.73863375370596
Iteration 7: 24.738633753705965
Iteration 8: 24.73863375370596

Actual Sqrt(612): 24.73863375370596


## 3. The Danger Zone (When Newton Fails)
Newton's method is fast, but it is **unstable**.
1.  **Stationary Points:** If $f'(x) \approx 0$ (flat slope), the term $\frac{f(x)}{f'(x)}$ becomes huge, shooting your guess to infinity.
2.  **Cycles:** Sometimes the method bounces back and forth between two points forever.

Because of this, scientific libraries use **Hybrid Methods** like **Brent's Method (`brentq`)**.
* **Brentq** combines Bisection (safety) + Secant Method (speed).
* It is the industry standard for root finding when you have a bracket.

## Problem 2: The Astronomer's Nightmare (Kepler's Equation)
**Context:** In orbital mechanics, we often need to find the position of a planet at a specific time. This requires solving **Kepler's Equation**:

$$M = E - e \sin(E)$$

Where:
* $M$ is the Mean Anomaly (Time-based, known).
* $e$ is the Eccentricity (Orbit shape, known).
* $E$ is the **Eccentric Anomaly** (Angle we need to find).

To solve for $E$, we must find the root of:
$$f(E) = E - e \sin(E) - M = 0$$

**Task:** Solve Kepler's equation for Earth's orbit using `scipy.optimize`.

In [ ]:
from scipy import optimize

# Constants for a hypothetical comet
M_target = 1.5   # We want to find E when M = 1.5 radians
ecc = 0.5        # Highly elliptical orbit

# 1. Define the function f(E)
# We want to find E such that: E - e*sin(E) - M = 0
def kepler_eqn(E):
    # TODO: Write the return statement for the equation above
    # Use np.sin(E)
    return E - ecc*np.sin(E) - M_target

# 2. Using Newton's Method (scipy.optimize.newton)
# Note: For 'newton', we usually provide the derivative (fprime) for speed,
# but Scipy can also estimate it if we omit it (Secant method).
def kepler_derivative(E):
    # Derivative of E is 1. Derivative of sin(E) is cos(E). M is constant.
    return 1 - ecc * np.cos(E)

# Guess E is roughly equal to M (often a good start for small e)
root_newton = optimize.newton(kepler_eqn, x0=M_target, fprime=kepler_derivative)
print(f"Newton Result (E): {root_newton:.6f}")


# 3. Using Brent's Method (scipy.optimize.brentq)
# Brentq requires a Bracket [a, b], not a single guess.
# Since E and M are angles, E must be somewhere between 0 and 2*Pi.
root_brent = optimize.brentq(kepler_eqn, a=0, b=2*np.pi)
print(f"Brentq Result (E): {root_brent:.6f}")

# --- AUTO-GRADER ---
assert np.isclose(root_newton, root_brent), "Methods disagree!"
# Verify the solution: Plug E back into M = E - e sin E
calc_M = root_brent - ecc * np.sin(root_brent)
print(f"Verification: Calculated M = {calc_M:.4f} (Expected {M_target})")
print("✅ Success! You solved the orbit.")

Newton Result (E): 1.962189
Brentq Result (E): 1.962189
Verification: Calculated M = 1.5000 (Expected 1.5)
✅ Success! You solved the orbit.


## Summary
1.  **Newton-Raphson** uses the derivative $f'(x)$ to converge quadratically fast.
2.  **Formula:** $x_{new} = x - f/f'$.
3.  **Risk:** It can explode if $f'(x) \to 0$.
4.  **Best Practice:** Use `scipy.optimize.brentq` if you can bracket the root (safest). Use `newton` if you have a good initial guess and the derivative (fastest).

# Unit 1: Gaussian Elimination & Linear Algebra
**Date:** Jan 23 (Friday)
**Topic:** Solving Systems of Linear Equations ($Ax = b$)

## 1. The Problem: $Ax = b$
In physics and engineering, we rarely solve just one equation. We usually solve systems of coupled equations.
For example, finding the currents in a circuit or the forces in a truss bridge requires solving a system like:

$$
\begin{cases}
2x + y - z = 8 \\
-3x - y + 2z = -11 \\
-2x + y + 2z = -3
\end{cases}
$$

This can be written in Matrix form $A \mathbf{x} = \mathbf{b}$:

$$
\underbrace{\begin{bmatrix} 2 & 1 & -1 \\ -3 & -1 & 2 \\ -2 & 1 & 2 \end{bmatrix}}_{A}
\underbrace{\begin{bmatrix} x \\ y \\ z \end{bmatrix}}_{\mathbf{x}} =
\underbrace{\begin{bmatrix} 8 \\ -11 \\ -3 \end{bmatrix}}_{\mathbf{b}}
$$

Our goal is to find the vector $\mathbf{x}$.

[Image of Gaussian elimination row operations visual guide]

## 2. The Method: Gaussian Elimination
While we could calculate the inverse ($x = A^{-1}b$), that is computationally expensive and unstable for large systems. instead, we use **Gaussian Elimination**.

### Step 1: Forward Elimination (Triangulation)
We use row operations to eliminate variables below the diagonal, turning $A$ into an **Upper Triangular Matrix**.
* Subtract multiples of the first row from rows below it to kill the $x$ terms.
* Subtract multiples of the second row from rows below it to kill the $y$ terms.

Target shape:
$$
\begin{bmatrix} \# & \# & \# \\ 0 & \# & \# \\ 0 & 0 & \# \end{bmatrix}
$$

### Step 2: Back Substitution
Once we have the triangle, the last equation is easy: $c z = d \Rightarrow z = d/c$.
We then plug $z$ into the row above to find $y$, and so on, moving up the matrix.

## 3. Worked Example (Manual Calculation)
Let's solve the system from above manually to understand the logic.

**System:**
1) $2x + 1y - 1z = 8$
2) $-3x - 1y + 2z = -11$
3) $-2x + 1y + 2z = -3$

**Step 1: Eliminate $x$ from Row 2 and Row 3**
* Target: Make $A_{2,0}$ and $A_{3,0}$ zero.
* Operation 1: $R_2 \leftarrow R_2 - (-1.5) \times R_1$. (Result: $0.5y + 0.5z = 1$)
* Operation 2: $R_3 \leftarrow R_3 - (-1) \times R_1$. (Result: $2y + 1z = 5$)

**Step 2: Eliminate $y$ from Row 3**
* Target: Make the new $A_{3,1}$ zero.
* Operation: $R_3 \leftarrow R_3 - (4) \times R_2$.
* Resulting equation: $(1 - 2)z = 5 - 4 \Rightarrow -1z = 1$.

**Step 3: Back Substitution**
* From $R_3$: $-z = 1 \Rightarrow \mathbf{z = -1}$
* From $R_2$: $0.5y + 0.5(-1) = 1 \Rightarrow 0.5y = 1.5 \Rightarrow \mathbf{y = 3}$
* From $R_1$: $2x + 3 - (-1) = 8 \Rightarrow 2x + 4 = 8 \Rightarrow 2x = 4 \Rightarrow \mathbf{x = 2}$

**Solution:** $(2, 3, -1)$

## 4. The Python Way: `numpy.linalg.solve`
Python does not require us to write the loop for Gaussian Elimination manually every time. We use the highly optimized LAPACK routines inside NumPy.

**Crucial Note:** Never use `inv(A)` to solve a system!
* `x = dot(inv(A), b)` is **slow** and **inaccurate**.
* `x = solve(A, b)` is **fast** and **stable**.

In [ ]:
import numpy as np

# Define the matrix A and vector b from the example
A = np.array([
    [ 2,  1, -1],
    [-3, -1,  2],
    [-2,  1,  2]
])

b = np.array([8, -11, -3])

# The Wrong Way (Inverse) - Just for demonstration
x_bad = np.linalg.inv(A).dot(b)

# The Right Way (Solve)
# Uses LU decomposition / Gaussian Elimination internally
x_sol = np.linalg.solve(A, b)
print(f"Solution x bad: {x_bad}")
print(f"Solution Vector x: {x_sol}")
print(f"Verify A @ x: {A @ x_sol}")
print(f"Original b:   {b}")

Solution x bad: [ 2.  3. -1.]
Solution Vector x: [ 2.  3. -1.]
Verify A @ x: [  8. -11.  -3.]
Original b:   [  8 -11  -3]


## Problem 1: The Circuit Breaker (Kirchhoff's Laws)
**Context:** You are analyzing a circuit with 3 loops. Using Kirchhoff's Voltage Law, you derive the following equations for currents $I_1, I_2, I_3$:

1. $50I_1 - 10I_2 - 20I_3 = 20$ (Volts)
2. $-10I_1 + 40I_2 - 10I_3 = 0$
3. $-20I_1 - 10I_2 + 60I_3 = 0$

**Task:** Set up the matrices and find the currents.

In [ ]:
# --- STUDENT SECTION ---

# TODO: Fill in the matrix A containing the resistance values
# Watch the signs!
R_matrix = np.array([
    [______, ______, ______],  # Equation 1 coefficients
    [______, ______, ______],  # Equation 2 coefficients
    [______, ______, ______]   # Equation 3 coefficients
])

# TODO: Fill in the voltage vector V
V_vector = np.array([______, ______, ______])

# TODO: Solve for I
Currents = ______

print(f"Currents (Amps): {Currents}")

# --- AUTO-GRADER ---
# Check simply: The sum of currents should be roughly 0.77 Amps
if Currents is not ...:
    assert np.isclose(np.sum(Currents), 0.773, atol=1e-3), "Currents seem incorrect. Check your matrix values."
    print("✅ Success! You solved the circuit.")

## Problem 2: The "Singular" Matrix Trap
**Context:** Not all systems have solutions. If two equations are parallel (linear dependent), the matrix is **Singular** (Determinant = 0).
Gaussian elimination will fail because it encounters a divide-by-zero.

**System:**
1) $x + y = 2$
2) $2x + 2y = 5$  (This is parallel to eq 1, but contradicts it. Impossible!/)

In [4]:
import numpy as np
# A singular matrix (Row 2 is 2x Row 1)
A_singular = np.array([
    [1, 1],
    [2, 2]
])

b_singular = np.array([2, 5])

print("Attempting to solve singular system...")

# x = np.linalg.solve(A_singular, b_singular)

try:
    # It should crash. That is the point of this exercise!
    x = np.linalg.solve(A_singular, b_singular)
    print("Result:", x)
except np.linalg.LinAlgError as e:
    print(f"\n✅ CAUGHT EXPECTED ERROR: {e}")
    print("Explanation: The matrix determinant is zero, so it has no unique solution.")
except Exception as e:
    print(f"Unexpected error: {e}")

Attempting to solve singular system...

✅ CAUGHT EXPECTED ERROR: Singular matrix
Explanation: The matrix determinant is zero, so it has no unique solution.
